# Lesson 5 — State Estimation & Robust Control

*ESP2110 Inverted Pendulum Lab*

**Run in Google Colab:** open the notebook, run the **Setup** cell once, then run
cells top-to-bottom. No local files are required.

## Learning objectives
By the end of this notebook you can:
1. Check **observability** — decide whether the full state can be reconstructed from the sensors you actually have.
2. Build a **Luenberger observer** to estimate the unmeasured velocities, and drive the controller from the *estimate*.
3. State and **verify the separation principle** — controller and observer poles can be chosen independently.
4. Replace naive numerical differentiation with a **Kalman filter** and see why it matters under measurement noise.
5. Stress-test the design against **parameter drift** (wrong pole length / cart mass).

### The setup that makes this lesson necessary
In Lessons 4A–4C the controller used the **full state** `x = [p, p_dot, theta, theta_dot]`.
On real hardware you do **not** measure velocities directly — you have a cart-position
encoder and a pole-angle sensor. So you measure only

$$y = \begin{bmatrix} p \\ \theta \end{bmatrix},\qquad C = \begin{bmatrix} 1&0&0&0 \\ 0&0&1&0 \end{bmatrix}.$$

The velocities must be **estimated**. That is what this lesson is about.

### Parameters (same plant as Lessons 4A/4B/4C)
| Symbol | Meaning | Value |
| --- | --- | --- |
| `m_c` | Cart mass | 0.5 kg |
| `m_p` | Pole mass | 0.2 kg |
| `L` | Pole length | 0.3 m |
| `g` | Gravity | 9.81 m/s^2 |
| `dt` | Sample time | 0.01 s |

In [ ]:
# --- Setup (safe to re-run) ---
try:
    import numpy, scipy, matplotlib  # noqa: F401
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'matplotlib'], check=True)
print('Environment ready.')

---
## Recap: the plant and the full-state controller

We reuse the exact plant and the full-state feedback gain `K` from Lesson 4B
(closed-loop poles placed at -2, -3, -4, -5). The only thing that changes is that
the controller can no longer see the whole state — it sees `y = [p, theta]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

m_c, m_p, L, g, dt = 0.5, 0.2, 0.3, 9.81, 0.01

def f_nonlin(s, f, Lt=L, mc=m_c, mp=m_p):
    """Full nonlinear cart-pole. Lt/mc/mp overridable for the drift study."""
    p, v, th, om = s
    sin, cos = np.sin(th), np.cos(th)
    den = mc + mp * sin**2
    vdot  = (f + mp * sin * (Lt * om**2 - g * cos)) / den
    omdot = (-f * cos - mp * Lt * om**2 * sin * cos + (mc + mp) * g * sin) / (Lt * den)
    return np.array([v, vdot, om, omdot])

A = np.array([
    [0, 1, 0,                       0],
    [0, 0, -m_p * g / m_c,          0],
    [0, 0, 0,                       1],
    [0, 0, (m_c + m_p) * g / (L * m_c), 0],
])
B = np.array([0, 1 / m_c, 0, -1 / (L * m_c)]).reshape(4, 1)

# Full-state feedback gain from Lesson 4B
K = signal.place_poles(A, B, [-2, -3, -4, -5]).gain_matrix.ravel()

# We measure ONLY cart position p and pole angle theta
C = np.array([[1, 0, 0, 0],
              [0, 0, 1, 0]])

print('K =', np.round(K, 3))
print('We measure y = C x =', ['p', 'theta'])

## Part 1 - Can you even reconstruct the state? (observability)

Before building an estimator, ask whether the measurements *contain enough information*
to reconstruct the full state. The test is the **observability matrix**

$$\mathcal{O} = \begin{bmatrix} C \\ CA \\ CA^2 \\ CA^3 \end{bmatrix}.$$

The system is observable **iff** `rank(O) = 4` (one row-block per state dimension).

In [ ]:
# TODO: build the observability matrix O by stacking C, C@A, C@A^2, C@A^3
#       (hint: np.linalg.matrix_power(A, i)), then print its shape and rank.
O = None  # <-- replace
# print('O shape', O.shape, ' rank', np.linalg.matrix_rank(O))


**Expected output.** `O` is an 8x4 matrix with **rank 4** -> the system is **observable**.
Measuring position and angle is enough to reconstruct the (unmeasured) velocities.
If you tried to measure *only* `p` or *only* `theta`, recompute the rank and see what happens.

## Part 2 - A Luenberger observer

The observer is a copy of the model that is continuously corrected by the measurement error:

$$\dot{\hat{x}} = A\hat{x} + Bf + L\,(y - C\hat{x}).$$

The estimation error `e = x - x_hat` obeys `e_dot = (A - LC) e`, so we choose `L` to place
the eigenvalues of `A - LC` in the left-half plane. **Rule of thumb:** make the observer
2-4x faster than the controller so the estimate converges before the controller leans on it.
We place observer poles at **-8, -10, -12, -14** (controller poles were -2,-3,-4,-5).

We compute `L` by pole placement on the *dual* system `(A^T, C^T)`: `L = place(A^T, C^T).T`.

In [ ]:
# TODO: (1) compute the observer gain L_obs by pole placement on the dual (A.T, C.T)
#           with observer poles [-8, -10, -12, -14], then L_obs = (...).gain_matrix.T
#       (2) write sim_output_feedback(T, x0, plant): run the observer
#           xh += dt*(A@xh + B.ravel()*f + L_obs@(y - C@xh)) and control f = -K@xh,
#           with y = C@x. Start xh at zeros, x0 default [0,0,0.1,0].
#       (3) plot true vs estimated theta and theta_dot.
L_obs = None  # <-- replace


**Expected output.** The estimated velocity starts at zero (the observer is ignorant) but
**locks onto the true velocity within a few tenths of a second**, after which the dashed and
solid curves overlap. The pole is stabilized using *only* `p` and `theta` measurements
(initial swing ~5.8 deg because the controller acts on a wrong estimate during the first instants).

### The separation principle (eigenvalue verification)

Stack the true state and the estimation error. The closed loop becomes

$$\begin{bmatrix}\dot{x}\\\dot{e}\end{bmatrix}=\begin{bmatrix}A-BK & BK\\ 0 & A-LC\end{bmatrix}\begin{bmatrix}x\\e\end{bmatrix}.$$

Because the matrix is block-triangular, its eigenvalues are exactly the **union** of the
controller poles `eig(A-BK)` and the observer poles `eig(A-LC)`. So you can design them
independently. Let's confirm it numerically.

In [ ]:
# Run after completing Part 2 to verify the separation principle.


## Part 3 - Measurement noise: naive differentiation vs the Kalman filter

Real sensors are noisy. A tempting shortcut is to get velocities by **finite-differencing**
the measured position and angle: `v ~= (y[k] - y[k-1]) / dt`. Differentiation **amplifies
noise** — dividing small random jitter by `dt = 0.01` blows it up 100x, and that noise goes
straight into the control force.

The **Kalman filter** is the optimal observer for known noise statistics: it is the
Luenberger observer with a gain that is continuously rebalanced between trusting the model
(`Q`) and trusting the measurement (`R`). Compare the two on the **control effort**.

In [ ]:
# TODO: implement run_noisy(method, seed, T) for method in {'naive','kalman'}.
#   - noisy measurement: y = C@x + meas_std*randn(2), meas_std = [0.005, radians(1)]
#   - 'naive': velocities by finite difference (y - y_prev)/dt, build xh=[p,vp,th,vth], f=-K@xh
#   - 'kalman': discrete predict/update with Q=diag([1e-6,1e-4,1e-6,1e-4]), R=diag(meas_std**2),
#               A_d = I + A*dt. f = -K@xh from the previous estimate.
#   Plot the control force for both and print the settled force std (t>1s) for each.
meas_std = np.array([0.005, np.radians(1.0)])
R = np.diag(meas_std**2); Q = np.diag([1e-6, 1e-4, 1e-6, 1e-4]); A_d = np.eye(4) + A * dt


**Expected output.** Both keep the pole near upright, but the **control force tells the real
story**: naive differentiation produces a force that **chatters with std ~7 N and peaks ~20 N**,
while the Kalman filter is **~30x smoother (std ~0.2 N, peak ~2 N)**. On hardware the chattering
naive controller would scream, overheat the motor, and wear the gears — even though the angle
plot looks "fine." *Smooth state estimates -> smooth control.*

## Part 4 - Robustness: what if your model is wrong?

We designed `K` and `L` for `L = 0.3 m`, `m_c = 0.5 kg`. Real parameters are never exact.
Sweep the *true* plant parameter while keeping the controller/observer fixed, and find where
the design stops working.

In [ ]:
# TODO: write survives(Lt, mc, T): run observer-based control on the nonlinear plant
#       with TRUE parameters Lt, mc (pass them into f_nonlin). Return max |theta| in
#       degrees, or None if it falls (|theta| > pi/2 or non-finite).
#       Then sweep pole length (0.7x..2.3x of 0.3) and cart mass (0.5x..3x of 0.5).


**Expected output.** The design is **very robust to pole length** — it tolerates roughly
0.7x to ~1.9x the design length, degrades badly by ~2.1x, and falls over past ~2.3x. It is
**much more sensitive to cart mass**: a cart only ~1.8x heavier than assumed already topples the pole. The lesson: robustness is
**parameter-specific**. Identify your most uncertain *and* most sensitive parameter and either
measure it carefully or design for the worst case.

## Part 5 - Watch it balance from estimates (animation)

The cart-pole below is stabilized using **only** position and angle measurements — the
controller never sees the true velocities. The faint pole is the observer's estimate; watch it
snap onto the true (solid) pole within the first moments.

In [ ]:
# Provided: run after Part 2 works to see observer-based balancing.
# (animation code is supplied in the Answer notebook)


---
## Checkpoints
- Observability matrix is 8x4 with **rank 4** -> the state is reconstructable from `p` and `theta`.
- The observer's estimated velocity converges to the true velocity within a few tenths of a second.
- Augmented closed-loop eigenvalues equal the **union** of controller poles {-2,-3,-4,-5} and observer poles {-8,-10,-12,-14} (separation principle).
- Under noise, the Kalman filter gives **dramatically smoother control force** than naive differentiation (~0.2 N vs ~7 N std).
- The design tolerates large pole-length error (to ~1.9x) but topples under a ~1.8x cart-mass error.

## Common pitfalls
- **Controlling on raw noisy measurements.** Finite-differencing position/angle for velocity
  injects huge noise into the force. Estimate, don't differentiate.
- **Observer too slow.** If observer poles are not faster than controller poles, the controller
  acts on a stale estimate and performance (or stability) degrades.
- **Wrong `L` construction.** `place_poles(A.T, C.T)` returns a gain whose **transpose** is `L`.
  Forgetting the `.T` gives wrong shapes/results.
- **Trusting the angle plot alone.** A controller can keep `theta` near zero while the *force*
  chatters destructively. Always inspect the control effort.
- **Assuming "robust to one parameter" means "robust."** Sensitivity is parameter-specific;
  check each uncertain parameter separately.